## 3. Caso de Automatización 
Para este punto entregar no solo el resultado, sino el archivo donde se desarrolló el proceso. 
El  uso  de  herramientas  de  Inteligencia  Artificial  es  bienvenido,  se  valorará  la  estructura, 
creatividad y recursividad. 
En  el  archivo  “withdrawals.xslx”  encontrará  4  pestañas,  correspondientes  a;  solicitudes  de 
retiro, snapshot de cuentas, actividad pendiente y registro de destinos. Con esa información, 
debe construir una lógica de decisión que determine, para cada solicitud, el estado del retiro: 
- APPROVE: Un retiro se aprueba si no cae en REJECT y no activa ninguna condición de 
HOLD. 
 
- HOLD: Un retiro deberá pasar por revisión manual si NO tiene razón para ser rechazado 
y cumple alguna de las siguientes condiciones: 
• available_cash - amount < BUFFER_USD 
• settled_cash - amount < BUFFER_USD (Ocurre riesgo de settlement: excede 
cash “settled”) 
• last_changed_at del destino está dentro de RECENT_DEST_DAYS 
• requested_speed = urgent y aml_risk_tier ∈ {medium, high} 
 
- REJECT: Un retiro será rechazado si: 
• account_status ≠ active 
• kyc_status ≠ verified 
• amount ≤ 0 
• Duplicado según la regla anterior 
• aml_risk_tier = high y is_whitelisted = false para ese destination_id 
 
**Reglas:**  
- El usuario debe mantener un balance mínimo de $50 luego de aprobar el retiro. 
BUFFER_USD = 50 
- La cuenta destino debe tener más de 8 días sin cambiarse, si el destino se cambió en 
los últimos 7 días, se revisa. RECENT_DEST_DAYS = 7  
- Se debe evitar enviar por error duplicados, un duplicado se define como mismo 
account_id + amount + destination_id dentro de 15 min. DUP_WINDOW_MIN = 15. 
- Las razones para cada decisión tendrán un puntaje de severidad, este servirá para 
priorizar qué casos se deben revisar primero. Si el retiro está ok, severidad es 0. 
Glosario de Razones (reason_code) y Severidad: 
 
- KYC_NOT_VERIFIED 100 
- ACCOUNT_NOT_ACTIVE 95 
- UNWHITELISTED_HIGH_AML 90 
- INVALID_AMOUNT 85 
- DUPLICATE_REQUEST 70 
- INSUFFICIENT_SETTLED_AFTER_BUFFER 65 
- INSUFFICIENT_AVAILABLE_AFTER_BUFFER 55 
- DEST_CHANGED_RECENTLY 45 
- URGENT_RISK_TIER 35 
### Aclaraciones: 
• Para “destino reciente”, comparar destination_registry.last_changed_at contra 
account_snapshot.as_of. 
• El archivo de review debe contener únicamente solicitudes con decisión HOLD y 
estar ordenado por severity de mayor a menor. 

3.1. Cree una base de datos de decisiones, donde cada request_id contenga su decisión, reason_code y severidad. 
(Bono) Una vez completado el ejercicio, proponga una idea para agentizar este proceso 
de forma escalable y eficiente. Mencione herramientas y flujo propuesto.

### 1. Parámetros

In [1]:
#  Parámetros globales 
BUFFER_USD       = 50   # Balance mínimo que debe quedar tras el retiro
RECENT_DEST_DAYS = 7    # Días mínimos sin cambios en el destino
DUP_WINDOW_MIN   = 15   # Ventana en minutos para detectar duplicados

# Severidad por reason_code
SEVERITY = {
    "KYC_NOT_VERIFIED":                    100,
    "ACCOUNT_NOT_ACTIVE":                   95,
    "UNWHITELISTED_HIGH_AML":               90,
    "INVALID_AMOUNT":                       85,
    "DUPLICATE_REQUEST":                    70,
    "INSUFFICIENT_SETTLED_AFTER_BUFFER":    65,
    "INSUFFICIENT_AVAILABLE_AFTER_BUFFER":  55,
    "DEST_CHANGED_RECENTLY":               45,
    "URGENT_RISK_TIER":                    35,
}

### 2. Carga de datos

In [2]:
# Importar las bibliotecas necesarias
import pandas as pd
from datetime import timedelta

FILE = "../files/withdrawals.xlsx"

requests = pd.read_excel(FILE, sheet_name="withdrawal_requests")
accounts = pd.read_excel(FILE, sheet_name="account_snapshot")
pending  = pd.read_excel(FILE, sheet_name="pending_activity")
registry = pd.read_excel(FILE, sheet_name="destination_registry")

# Convertir fechas a datetime con timezone
requests["created_at"]      = pd.to_datetime(requests["created_at"], utc=True)
accounts["as_of"]           = pd.to_datetime(accounts["as_of"], utc=True)
registry["last_changed_at"] = pd.to_datetime(registry["last_changed_at"], utc=True)

print(f"Solicitudes de retiro : {len(requests)}")
print(f"Cuentas en snapshot   : {len(accounts)}")
print(f"Actividad pendiente   : {len(pending)}")
print(f"Destinos registrados  : {len(registry)}")

Solicitudes de retiro : 128
Cuentas en snapshot   : 35
Actividad pendiente   : 99
Destinos registrados  : 52


In [4]:
# Validaciones de datos para no tener NaNs o tipos incorrectos
missing_accounts = requests[~requests["account_id"].isin(accounts["account_id"])]
if len(missing_accounts) > 0:
    print(f"{len(missing_accounts)} solicitudes sin cuenta asociada")
else:
    print(" Todas las solicitudes tienen cuenta registrada")

# Validar destination_id
missing_destinations = requests[~requests["destination_id"].isin(registry["destination_id"])]
if len(missing_destinations) > 0:
    print(f" {len(missing_destinations)} solicitudes con destino no registrado")
else:
    print(" Todos los destinos están registrados")

 Todas las solicitudes tienen cuenta registrada
 Todos los destinos están registrados


### 3. Unión de los datos

In [6]:
# Unir solicitudes con snapshot de cuenta
df = requests.merge(accounts, on="account_id", how="left")

# Unir con registro de destinos
df = df.merge(
    registry[["destination_id", "is_whitelisted", "last_changed_at"]],
    on="destination_id",
    how="left"
)
df[["request_id", "account_id", "amount", "account_status",
    "kyc_status", "aml_risk_tier", "available_cash",
    "settled_cash", "is_whitelisted", "last_changed_at"]].head()

,request_id,account_id,amount,account_status,kyc_status,aml_risk_tier,available_cash,settled_cash,is_whitelisted,last_changed_at
0,W100055,A0032,2000.27,active,verified,low,12529.37,11684.78,True,2025-06-27 09:00:00+00:00
1,W100040,A0008,4048.65,active,verified,low,20444.37,16255.79,True,2025-04-03 16:00:00+00:00
2,W100019,A0014,3352.79,frozen,verified,low,15815.23,15815.23,True,2026-03-04 11:00:00+00:00
3,W100031,A0027,26978.71,closed,verified,medium,114617.12,91487.88,True,2025-11-27 00:00:00+00:00
4,W100098,A0023,2861.14,active,verified,high,39952.75,33268.86,True,2025-09-17 15:00:00+00:00


### 4. Decisiones a tomar

Aplicamos las reglas en orden de prioridad:
1. Primero chequeamos razones de REJECT
2. Luego chequeamos condiciones de HOLD
3. Si no cae en ninguna se procede con APPROVE

In [7]:
def evaluar_retiro(row, df_completo):
    reasons = []  # Lista de (reason_code, decision) encontradas

    # REJECT

    # 1. Cuenta no activa
    if row["account_status"] != "active":
        reasons.append(("ACCOUNT_NOT_ACTIVE", "REJECT"))

    # 2. KYC no verificado
    if row["kyc_status"] != "verified":
        reasons.append(("KYC_NOT_VERIFIED", "REJECT"))

    # 3. Monto inválido
    if row["amount"] <= 0:
        reasons.append(("INVALID_AMOUNT", "REJECT"))

    # 4. Duplicado: mismo account_id + amount + destination_id dentro de 15 min
    ventana = timedelta(minutes=DUP_WINDOW_MIN)
    duplicados = df_completo[
        (df_completo["account_id"]     == row["account_id"]) &
        (df_completo["amount"]         == row["amount"]) &
        (df_completo["destination_id"] == row["destination_id"]) &
        (df_completo["request_id"]     != row["request_id"]) &
        (abs(df_completo["created_at"] - row["created_at"]) <= ventana)
    ]
    if not duplicados.empty:
        reasons.append(("DUPLICATE_REQUEST", "REJECT"))

    # 5. AML alto y destino no whitelisted
    if row["aml_risk_tier"] == "high" and row["is_whitelisted"] == False:
        reasons.append(("UNWHITELISTED_HIGH_AML", "REJECT"))

    # HOLD

    if not any(d == "REJECT" for _, d in reasons):

        # 6. Saldo disponible insuficiente tras el retiro
        if row["available_cash"] - row["amount"] < BUFFER_USD:
            reasons.append(("INSUFFICIENT_AVAILABLE_AFTER_BUFFER", "HOLD"))

        # 7. Saldo liquidado insuficiente tras el retiro
        if row["settled_cash"] - row["amount"] < BUFFER_USD:
            reasons.append(("INSUFFICIENT_SETTLED_AFTER_BUFFER", "HOLD"))

        # 8. Destino modificado recientemente
        if pd.notna(row["last_changed_at"]):
            dias_desde_cambio = (row["as_of"] - row["last_changed_at"]).days
            if dias_desde_cambio < RECENT_DEST_DAYS:
                reasons.append(("DEST_CHANGED_RECENTLY", "HOLD"))

        # 9. Retiro urgente con riesgo AML medio o alto
        if row["requested_speed"] == "urgent" and row["aml_risk_tier"] in {"medium", "high"}:
            reasons.append(("URGENT_RISK_TIER", "HOLD"))

    # Determinar decisión final
    if not reasons:
        return "APPROVE", "NONE", 0

    # Nos quedamos con la razón de mayor severidad
    reasons_sorted = sorted(reasons, key=lambda x: SEVERITY[x[0]], reverse=True)
    top_reason, top_decision = reasons_sorted[0]

    return top_decision, top_reason, SEVERITY[top_reason]

### 5. Aplicar la función de toma de decisiones

In [8]:
# Aplicamos la función a cada fila
resultados = df.apply(lambda row: evaluar_retiro(row, df), axis=1, result_type="expand")
resultados.columns = ["decision", "reason_code", "severity"]

# Base de datos de decisiones (3.1)
decisions_db = pd.concat([df[["request_id", "account_id", "amount"]], resultados], axis=1)

print("Decisiones calculadas")
print()
print(decisions_db["decision"].value_counts().to_string())

Decisiones calculadas

decision
APPROVE    51
REJECT     42
HOLD       35


In [9]:
# Vista completa de la base de decisiones
print("BASE DE DATOS CON DECISIONES")
decisions_db.sort_values("severity", ascending=False).head(20)

BASE DE DATOS CON DECISIONES


,request_id,account_id,amount,decision,reason_code,severity
114,W100087,A0013,5836.91,REJECT,KYC_NOT_VERIFIED,100
107,W100001,A0016,3049.90,REJECT,KYC_NOT_VERIFIED,100
126,W100051,A0012,18378.93,REJECT,KYC_NOT_VERIFIED,100
120,W100020,A0012,5257.29,REJECT,KYC_NOT_VERIFIED,100
24,W100105,A0022,992.26,REJECT,KYC_NOT_VERIFIED,100
118,W100082,A0012,4593.63,REJECT,KYC_NOT_VERIFIED,100
83,W100050,A0016,2844.67,REJECT,KYC_NOT_VERIFIED,100
68,W100112,A0006,15791.24,REJECT,ACCOUNT_NOT_ACTIVE,95
58,W100039,A0014,1859.07,REJECT,ACCOUNT_NOT_ACTIVE,95
26,W100070,A0001,16280.18,REJECT,ACCOUNT_NOT_ACTIVE,95


### 6. Revisión de HOLD 

In [10]:
review_file = decisions_db[decisions_db["decision"] == "HOLD"].sort_values(
    "severity", ascending=False
).reset_index(drop=True)

print(f"Solicitudes en HOLD para revisión manual: {len(review_file)}")
print()
print(review_file.to_string())

Solicitudes en HOLD para revisión manual: 35

   request_id account_id    amount decision                        reason_code  severity
0     W100069      A0020   8906.77     HOLD  INSUFFICIENT_SETTLED_AFTER_BUFFER        65
1     W100054      A0002   6719.15     HOLD  INSUFFICIENT_SETTLED_AFTER_BUFFER        65
2     W100072      A0020   8669.22     HOLD  INSUFFICIENT_SETTLED_AFTER_BUFFER        65
3     W100058      A0018  10730.44     HOLD  INSUFFICIENT_SETTLED_AFTER_BUFFER        65
4     W100109      A0028   7110.68     HOLD  INSUFFICIENT_SETTLED_AFTER_BUFFER        65
5     W100032      A0010   9330.97     HOLD  INSUFFICIENT_SETTLED_AFTER_BUFFER        65
6     W100005      A0028   6859.80     HOLD  INSUFFICIENT_SETTLED_AFTER_BUFFER        65
7     W100029      A0003  27278.01     HOLD  INSUFFICIENT_SETTLED_AFTER_BUFFER        65
8     W100052      A0004   9835.49     HOLD  INSUFFICIENT_SETTLED_AFTER_BUFFER        65
9     W100022      A0019  11612.55     HOLD  INSUFFICIENT_SETTLE

### 7. Data obtenida final: Decisiones y revisión de HOLD

In [11]:
# Exportar resultados
decisions_db.to_csv("decisions_db.csv", index=False)
review_file.to_csv("review_hold.csv", index=False)

print("Archivos exportados:")
print("  decisions_db.csv  — todas las decisiones")
print("  review_hold.csv   — solo HOLDs para revisión manual")

Archivos exportados:
  decisions_db.csv  — todas las decisiones
  review_hold.csv   — solo HOLDs para revisión manual


## Bonus

Para escalar este proceso propongo una arquitectura orientada a eventos. 
Básicamente, en vez de correr un script manualmente, cada vez que llega una solicitud nueva se dispara todo automáticamente.
Para recibir los mensajes usaría RabbitMQ como cola de mensajes, se van acumulando las solicitudes y el agente las procesa una por una sin perder ninguna. Si llegan varias a la vez, no habría problema.
El agente está construido con la API de Claude y tiene tres herramientas: consulta el estado de la cuenta, verifica el destino y detecta duplicados. Para los duplicados usaría Redis, que es una base de datos en memoria, la cual es ideal para revisar si la misma solicitud ya llegó en los últimos 15 minutos.
Las decisiones las guardo en MongoDB porque cada decisión es básicamente un documento JSON con campos variables, y Mongo maneja eso de forma nativa.
Cuando hay un HOLD, en vez de revisar un archivo Excel, le llega una alerta automática al equipo por Slack con el detalle del caso y la severidad, para saber cuál revisar primero.
Lo mejor de esta arquitectura es que escala horizontalmente, si el volumen crece, simplemente levanto más instancias del agente consumiendo la misma cola, sin tocar nada del código.